# 05 — ARIMA na prática

No notebook anterior, aplicamos diferenciação e vimos que `d = 1` é um bom ponto de partida.

Agora vamos treinar o primeiro modelo ARIMA.

Neste notebook, vamos:

- preparar a série para modelagem;
- separar treino e teste respeitando o tempo;
- criar uma baseline simples;
- treinar um modelo ARIMA;
- gerar previsões;
- avaliar o erro do modelo.

### 05.01 Imports e caminhos

In [15]:
from pathlib import Path
import sys

import pandas as pd
import numpy as np

import plotly.graph_objects as go

from statsmodels.tsa.arima.model import ARIMA

from sklearn.metrics import mean_absolute_error, mean_squared_error

if Path.cwd().name == "notebooks":
    raiz_projeto = Path.cwd().parent
else:
    raiz_projeto = Path.cwd()

sys.path.append(str(raiz_projeto / "src"))

from visual_utils import (
    CORES,
    aplicar_layout_padrao,
    grafico_linha_padrao
)

caminho_serie_diaria = raiz_projeto / "outputs" / "tabelas" / "serie_diaria_bike.csv"
caminho_outputs_tabelas = raiz_projeto / "outputs" / "tabelas"

print("Série diária:")
print(caminho_serie_diaria)

Série diária:
c:\GitHub\ALURA-TESTE\series-temporais-bikes\outputs\tabelas\serie_diaria_bike.csv


### 05.02 Carregar série diária

In [16]:
serie_diaria = pd.read_csv(caminho_serie_diaria)

serie_diaria["data_hora"] = pd.to_datetime(serie_diaria["data_hora"])

serie_diaria = serie_diaria.sort_values("data_hora").reset_index(drop=True)

print(f"Linhas: {serie_diaria.shape[0]}")
print(f"Colunas: {serie_diaria.shape[1]}")

serie_diaria.head()

Linhas: 456
Colunas: 2


,data_hora,demanda
0,2011-01-01,985
1,2011-01-02,801
2,2011-01-03,1349
3,2011-01-04,1562
4,2011-01-05,1600


## Preparando a série

Para o ARIMA, vamos trabalhar com a série de demanda diária.

A variável que queremos prever é `demanda`.

### 05.03 Criar série de modelagem

In [17]:
serie_modelagem = serie_diaria[["data_hora", "demanda"]].copy()

serie_modelagem.head()

,data_hora,demanda
0,2011-01-01,985
1,2011-01-02,801
2,2011-01-03,1349
3,2011-01-04,1562
4,2011-01-05,1600


### 05.04 Visualizar série

In [18]:
fig = grafico_linha_padrao(
    df=serie_modelagem,
    x="data_hora",
    y="demanda",
    titulo="Série de demanda diária para modelagem",
    labels={
        "data_hora": "Data",
        "demanda": "Demanda diária"
    },
    altura=500
)

fig.show()

## Treino e teste temporal

Em séries temporais, não embaralhamos os dados.

O treino representa o passado.

O teste representa um período futuro que o modelo ainda não viu.

### 05.05 Separar treino e teste

In [19]:
tamanho_teste = 60

treino = serie_modelagem.iloc[:-tamanho_teste].copy()
teste = serie_modelagem.iloc[-tamanho_teste:].copy()

print("Tamanho do treino:", treino.shape[0])
print("Tamanho do teste:", teste.shape[0])

print("\nPeríodo de treino:")
print(treino["data_hora"].min(), "até", treino["data_hora"].max())

print("\nPeríodo de teste:")
print(teste["data_hora"].min(), "até", teste["data_hora"].max())

Tamanho do treino: 396
Tamanho do teste: 60

Período de treino:
2011-01-01 00:00:00 até 2012-09-16 00:00:00

Período de teste:
2012-09-17 00:00:00 até 2012-12-19 00:00:00


### 05.06 Visualizar treino e teste

In [20]:
fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=treino["data_hora"],
        y=treino["demanda"],
        mode="lines",
        name="Treino",
        line=dict(color=CORES["azul_principal"], width=3)
    )
)

fig.add_trace(
    go.Scatter(
        x=teste["data_hora"],
        y=teste["demanda"],
        mode="lines",
        name="Teste",
        line=dict(color=CORES["azul_claro"], width=3)
    )
)

fig = aplicar_layout_padrao(
    fig,
    titulo="Separação temporal entre treino e teste",
    altura=500
)

fig.show()

## Baseline simples

Antes de avaliar o ARIMA, precisamos de uma comparação mínima.

A baseline será uma regra simples: usar o último valor observado no treino como previsão para todo o período de teste.

### 05.07 Criar baseline

In [21]:
ultimo_valor_treino = treino["demanda"].iloc[-1]

teste["previsao_baseline"] = ultimo_valor_treino

teste[["data_hora", "demanda", "previsao_baseline"]].head()

,data_hora,demanda,previsao_baseline
396,2012-09-17,6869,7333
397,2012-09-18,4073,7333
398,2012-09-19,7591,7333
399,2012-10-01,6778,7333
400,2012-10-02,4639,7333


## Treinando o ARIMA

Vamos começar com um modelo ARIMA simples.

Como aplicamos uma diferenciação no notebook anterior, vamos usar `d = 1`.

Um primeiro ponto de partida será:

`ARIMA(1, 1, 1)`

### 05.08 Treinar modelo ARIMA

In [22]:
ordem_arima = (1, 1, 1)

modelo_arima = ARIMA(
    treino["demanda"],
    order=ordem_arima
)

resultado_arima = modelo_arima.fit()

resultado_arima.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                               SARIMAX Results                                
==============================================================================
Dep. Variable:                demanda   No. Observations:                  396
Model:                 ARIMA(1, 1, 1)   Log Likelihood               -3205.271
Date:                Fri, 26 Jun 2026   AIC                           6416.542
Time:                        10:56:35   BIC                           6428.479
Sample:                             0   HQIC                          6421.272
                                - 396                                         
Covariance Type:                  opg                                         
==============================================================================
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
ar.L1          0.2500      0.052      4.834      0.000       0.149       0.351
ma.L1         -0.8480      0.030    -28.166      0.000      -0.907      -0.789
sigma2      6.529e+05   3.46e+04     18.854      0.000    5.85e+05    7.21e+05
===================================================================================
Ljung-Box (L1) (Q):                   0.02   Jarque-Bera (JB):               119.78
Prob(Q):                              0.89   Prob(JB):                         0.00
Heteroskedasticity (H):               2.68   Skew:                            -0.77
Prob(H) (two-sided):                  0.00   Kurtosis:                         5.22
===================================================================================

Warnings:
[1] Covariance matrix calculated using the outer product of gradients (complex-step).
"""

### 05.09 Gerar previsões

In [23]:
previsao_arima = resultado_arima.forecast(steps=len(teste))

teste["previsao_arima"] = previsao_arima.values

teste[["data_hora", "demanda", "previsao_baseline", "previsao_arima"]].head()

,data_hora,demanda,previsao_baseline,previsao_arima
396,2012-09-17,6869,7333,7486.108513
397,2012-09-18,4073,7333,7524.392858
398,2012-09-19,7591,7333,7533.965749
399,2012-10-01,6778,7333,7536.359422
400,2012-10-02,4639,7333,7536.957954


### 05.10 Visualizar real versus previsto

In [24]:
fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=teste["data_hora"],
        y=teste["demanda"],
        mode="lines",
        name="Real",
        line=dict(color=CORES["azul_escuro"], width=3)
    )
)

fig.add_trace(
    go.Scatter(
        x=teste["data_hora"],
        y=teste["previsao_baseline"],
        mode="lines",
        name="Baseline",
        line=dict(color=CORES["cinza_suave"], width=2, dash="dash")
    )
)

fig.add_trace(
    go.Scatter(
        x=teste["data_hora"],
        y=teste["previsao_arima"],
        mode="lines",
        name="ARIMA",
        line=dict(color=CORES["azul_principal"], width=3)
    )
)

fig = aplicar_layout_padrao(
    fig,
    titulo="Real versus previsto — Baseline e ARIMA",
    altura=500
)

fig.show()

## Métricas de erro

Vamos avaliar os modelos usando três métricas:

- MAE: erro médio absoluto;
- RMSE: erro que penaliza mais erros grandes;
- MAPE: erro percentual médio.

### 05.11 Criar função de métricas

In [25]:
def calcular_metricas(y_real, y_pred):
    mae = mean_absolute_error(y_real, y_pred)
    rmse = np.sqrt(mean_squared_error(y_real, y_pred))

    y_real = np.array(y_real)
    y_pred = np.array(y_pred)

    mascara = y_real != 0
    mape = np.mean(
        np.abs((y_real[mascara] - y_pred[mascara]) / y_real[mascara])
    ) * 100

    return {
        "MAE": mae,
        "RMSE": rmse,
        "MAPE": mape
    }

### 05.12 Calcular métricas

In [26]:
metricas_baseline = calcular_metricas(
    teste["demanda"],
    teste["previsao_baseline"]
)

metricas_arima = calcular_metricas(
    teste["demanda"],
    teste["previsao_arima"]
)

df_metricas_arima = pd.DataFrame([
    {
        "modelo": "Baseline",
        **metricas_baseline
    },
    {
        "modelo": "ARIMA(1,1,1)",
        **metricas_arima
    }
])

df_metricas_arima

,modelo,MAE,RMSE,MAPE
0,Baseline,1585.183333,1865.908943,31.813723
1,"ARIMA(1,1,1)",1730.649703,2031.357449,34.679064


### 05.13 Resíduos do ARIMA

In [27]:
teste["residuo_arima"] = teste["demanda"] - teste["previsao_arima"]

fig = grafico_linha_padrao(
    df=teste,
    x="data_hora",
    y="residuo_arima",
    titulo="Resíduos do modelo ARIMA no período de teste",
    labels={
        "data_hora": "Data",
        "residuo_arima": "Erro"
    },
    altura=450
)

fig.show()

### 05.14 Salvar resultados

In [28]:
caminho_previsoes_arima = caminho_outputs_tabelas / "previsoes_arima.csv"
caminho_metricas_arima = caminho_outputs_tabelas / "metricas_arima.csv"

teste.to_csv(
    caminho_previsoes_arima,
    index=False,
    encoding="utf-8-sig"
)

df_metricas_arima.to_csv(
    caminho_metricas_arima,
    index=False,
    encoding="utf-8-sig"
)

print("Arquivos salvos:")
print("-", caminho_previsoes_arima)
print("-", caminho_metricas_arima)

Arquivos salvos:
- c:\GitHub\ALURA-TESTE\series-temporais-bikes\outputs\tabelas\previsoes_arima.csv
- c:\GitHub\ALURA-TESTE\series-temporais-bikes\outputs\tabelas\metricas_arima.csv


### 05.15 Resultado parcial

In [29]:
melhor_modelo_ate_agora = df_metricas_arima.sort_values("MAPE").iloc[0]

print("Melhor modelo até agora:")
print(melhor_modelo_ate_agora["modelo"])

print("\nMAPE:")
print(f"{melhor_modelo_ate_agora['MAPE']:.2f}%")

Melhor modelo até agora:
Baseline

MAPE:
31.81%


## Leitura parcial

Neste primeiro teste, o ARIMA(1,1,1) não superou a baseline.

Isso mostra que um modelo mais complexo não é automaticamente melhor.

No próximo notebook, vamos testar o SARIMA, que adiciona uma camada sazonal ao modelo.

## Próximo passo

Treinamos o primeiro ARIMA e comparamos seu desempenho com uma baseline simples.

No próximo notebook, vamos investigar se um modelo com componente sazonal, o SARIMA, consegue representar melhor os ciclos da série.